In [0]:
%sql
CREATE OR REPLACE TEMP VIEW order_reviews_typed AS
SELECT
  review_id
  , order_id
  , TRY_CAST(review_score AS INTEGER) as review_score
  , review_comment_title
  , review_comment_message
  , TRY_CAST(review_creation_date AS DATE) as review_creation_date
  , TRY_CAST(review_answer_timestamp AS TIMESTAMP) as review_answer_timestamp
  , _ingested_at
FROM workspace.bronze.order_reviews
QUALIFY ROW_NUMBER() OVER (PARTITION BY review_id, order_id ORDER BY _ingested_at DESC) = 1;

CREATE OR REPLACE TEMP VIEW order_reviews_validated AS
SELECT *,
  CASE
    WHEN review_id IS NULL THEN 'llave nula'
    WHEN order_id IS NULL THEN 'llave nula'
    WHEN review_score IS NULL OR review_score NOT BETWEEN 1 AND 5 THEN 'score invalido'
    WHEN review_answer_timestamp < review_creation_date THEN 'tiempo de respusta invalido'
  END AS motivo
FROM order_reviews_typed;

CREATE OR REPLACE TABLE workspace.silver.order_reviews
AS
SELECT * EXCEPT(motivo) FROM order_reviews_validated
WHERE motivo IS NULL;

CREATE OR REPLACE TABLE workspace.silver.order_reviews_quarantine
AS
SELECT * FROM order_reviews_validated
WHERE motivo IS NOT NULL;